In [ ]:
# If needed, uncomment and run once, then restart kernel.
!pip install -U transformers torch tqdm pandas pyarrow

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

# Resolve the data path robustly (works when the kernel starts in project root or in 4_Embedding_Extraction)
CANDIDATES = [
    Path("../1_Data/preprocessed_data.csv"),  # expected when notebook is in 4_Embedding_Extraction
    Path("1_Data/preprocessed_data.csv"),     # expected when kernel CWD is the project root
]
for p in CANDIDATES:
    if p.exists():
        CSV_PATH = p
        break
else:
    raise FileNotFoundError("Could not find preprocessed_data.csv. Checked: "
                            + ", ".join(str(p.resolve()) for p in CANDIDATES))

OUT_DIR = CSV_PATH.parent  # ../1_Data
print("CSV_PATH:", CSV_PATH.resolve())
print("Saving outputs to:", OUT_DIR.resolve())


In [ ]:
df = pd.read_csv(CSV_PATH)
expected = {"Clone name", "VH Protein", "VL Protein"}
missing = expected - set(df.columns)
assert not missing, f"CSV missing columns: {missing}"

def clean_seq(s: str) -> str:
    """Uppercase; remove spaces/dashes; convert non‑std AA to 'X'."""
    if pd.isna(s):
        return ""
    s = str(s).upper().strip()
    s = re.sub(r"[\s\-.]", "", s)
    # Keep 20 canonical AAs + X (unknown); map everything else to X
    allowed = set("ACDEFGHIKLMNPQRSTVWY")
    s = "".join(ch if ch in allowed else "X" for ch in s)
    return s

df["VH_clean"] = df["VH Protein"].map(clean_seq)
df["VL_clean"] = df["VL Protein"].map(clean_seq)

# Drop rows with empty sequences (defensive)
before = len(df)
df = df[(df["VH_clean"].str.len() > 0) & (df["VL_clean"].str.len() > 0)].reset_index(drop=True)
after = len(df)
if after < before:
    print(f"Dropped {before - after} rows with empty VH/VL after cleaning.")

clone_names = df["Clone name"].tolist()
vh_sequences = df["VH_clean"].tolist()
vl_sequences = df["VL_clean"].tolist()

print(f"Loaded {len(df)} clones. Example VH/VL lengths: {len(vh_sequences[0])}/{len(vl_sequences[0])}")

In [ ]:
model_name = "facebook/esm2_t33_650M_UR50D"  # HF model id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval().to(device)

EMBED_DIM = model.config.hidden_size  # 640 for t30_150M
print("Embed dim:", EMBED_DIM)

In [ ]:
@torch.no_grad()
def embed_sequences_mean(seq_list, batch_size=32):
    """
    Returns: np.ndarray of shape (N, EMBED_DIM).
    Mean-pooled over residue tokens; special tokens ([CLS]/<bos>, <eos>, etc.) are excluded.
    """
    all_vecs = []

    for i in tqdm(range(0, len(seq_list), batch_size), desc="Embedding"):
        batch = seq_list[i:i+batch_size]

        toks = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,          # pad to longest in batch
            truncation=True,       # (not needed for short V regions, but safe)
            add_special_tokens=True
        )
        toks = {k: v.to(device) for k, v in toks.items()}

        out = model(**toks)  # last_hidden_state: (B, T, D)
        hidden = out.last_hidden_state
        input_ids = toks["input_ids"]
        attn = toks["attention_mask"]

        # Build a mask that excludes special tokens (CLS/BOS/EOS/PAD)
        mask = attn.clone()
        special_ids = set(tokenizer.all_special_ids)  # includes pad/cls/eos, etc.
        for sid in special_ids:
            mask = mask * (input_ids != sid)

        mask_f = mask.float().unsqueeze(-1)          # (B, T, 1)
        # Avoid division by zero (should not happen with non-empty sequences)
        denom = mask.sum(dim=1).clamp(min=1).unsqueeze(-1).float()  # (B, 1)

        pooled = (hidden * mask_f).sum(dim=1) / denom  # (B, D)
        all_vecs.append(pooled.detach().cpu().numpy())

        # Free memory
        del out, hidden, toks

    return np.concatenate(all_vecs, axis=0)

In [ ]:
BATCH_SIZE = 32 if device.type == "cuda" else 8

vh_embeddings = embed_sequences_mean(vh_sequences, batch_size=BATCH_SIZE)
vl_embeddings = embed_sequences_mean(vl_sequences, batch_size=BATCH_SIZE)

print("VH embeddings:", vh_embeddings.shape)
print("VL embeddings:", vl_embeddings.shape)
assert vh_embeddings.shape == vl_embeddings.shape == (len(df), EMBED_DIM)

In [ ]:
# 6a) Compact NPZ bundle
npz_path = OUT_DIR / "hf_esm2_t33_650M_embeddings_vh_vl.npz"
np.savez_compressed(
    npz_path,
    model_name=model_name,
    embed_dim=EMBED_DIM,
    clone_names=np.array(clone_names, dtype=object),
    vh_embeddings=vh_embeddings,
    vl_embeddings=vl_embeddings,
)
print("Saved NPZ:", npz_path.resolve())

# 6b) Parquet (fast & compact) and CSV (wide files but convenient)
vh_df = pd.DataFrame(vh_embeddings, index=clone_names, columns=[f"vh_{i}" for i in range(EMBED_DIM)]).reset_index().rename(columns={"index": "Clone name"})
vl_df = pd.DataFrame(vl_embeddings, index=clone_names, columns=[f"vl_{i}" for i in range(EMBED_DIM)]).reset_index().rename(columns={"index": "Clone name"})

vh_parq = OUT_DIR / "hf_esm2_t33_650M_vh_embeddings.parquet"
vl_parq = OUT_DIR / "hf_esm2_t33_650M_vl_embeddings.parquet"
vh_df.to_parquet(vh_parq, index=False)
vl_df.to_parquet(vl_parq, index=False)
print("Saved Parquet:\n ", vh_parq.resolve(), "\n ", vl_parq.resolve())

# Optional CSVs
vh_csv = OUT_DIR / "hf_esm2_t33_650M_vh_embeddings.csv"
vl_csv = OUT_DIR / "hf_esm2_t33_650M_vl_embeddings.csv"
vh_df.to_csv(vh_csv, index=False)
vl_df.to_csv(vl_csv, index=False)
print("Saved CSV:\n ", vh_csv.resolve(), "\n ", vl_csv.resolve())

In [ ]:
combined = np.concatenate([vh_embeddings, vl_embeddings], axis=1)  # shape: (N, 2*EMBED_DIM)
combined_df = pd.DataFrame(combined, index=clone_names,
                           columns=[f"vh_{i}" for i in range(EMBED_DIM)] + [f"vl_{i}" for i in range(EMBED_DIM)]).reset_index().rename(columns={"index": "Clone name"})

comb_parq = OUT_DIR / "hf_esm2_t33_650M_vhvl_embeddings.parquet"
comb_csv  = OUT_DIR / "hf_esm2_t33_650M_vhvl_embeddings.csv"
combined_df.to_parquet(comb_parq, index=False)
combined_df.to_csv(comb_csv, index=False)
print("Saved combined embeddings:\n ", comb_parq.resolve(), "\n ", comb_csv.resolve())